In [74]:
from IPython.display import Markdown, display
import os
from tqdm import tqdm
from yandex_cloud_ml_sdk import YCloudML
from glob import glob
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path
import json
import numpy as np


def printx(string):
    display(Markdown(string))

def create_assistant(model, tools=None):
    kwargs = {}
    if tools and len(tools) > 0:
        kwargs = {"tools": tools}
    return sdk.assistants.create(
        model, ttl_days=1, expiration_policy="since_last_active", **kwargs
    )

folder_id = 'b1gst3c7cskk2big5fqn'
api_key = 'AQVNzzJielnSayrAOlQWlxDMK49OShvzdqtUQdAp'

sdk = YCloudML(folder_id=folder_id, auth=api_key)
model = (
  sdk.models.text_embeddings(model_name="text-search-query", model_version="latest")
)

In [75]:
import json
import pandas as pd
from pathlib import Path


json_dir = '/Users/ogzeus/Downloads/'
json_files = list(Path(json_dir).glob('visual-2.txt'))
print(json_files)
all_data = []
for json_file in json_files:
    with open(json_file, 'r', encoding='utf-8') as file:
        data = file.readlines()
        if isinstance(data, list):
            all_data.extend(data)
        else:
            all_data.append(data)
df = pd.DataFrame(all_data)

[PosixPath('/Users/ogzeus/Downloads/visual-2.txt')]


In [76]:
# Фильтруем ненулевые элементы и создаем DataFrame за один шаг
filtered_data = [item for item in df[0] if item != '']
new_df = pd.DataFrame(filtered_data, columns=["Data"])

In [79]:
embeds = []
for i in tqdm(new_df['Data']):
    res = model.run(i)
    embeds.append(res)

  0%|          | 0/1617 [00:00<?, ?it/s]

AioRpcError: <AioRpcError of RPC that terminated with:
	code = StatusCode.INVALID_ARGUMENT
	details = "Error in session internal_id=1deb4a7a-b17e5e70-bc2438cb-6dc8efa0&request_id=f059d111-001f-4bee-ac85-c054eec71ec7&client_request_id=13c4a031-8802-4fe2-abb6-976bbed018bd&folder_id=b1giflegi2ke62rs80la: number of input tokens must be no more than 2048, got 2902"
	debug_error_string = "UNKNOWN:Error received from peer ipv4:158.160.54.160:443 {created_time:"2025-05-01T01:10:12.669146+03:00", grpc_status:3, grpc_message:"Error in session internal_id=1deb4a7a-b17e5e70-bc2438cb-6dc8efa0&request_id=f059d111-001f-4bee-ac85-c054eec71ec7&client_request_id=13c4a031-8802-4fe2-abb6-976bbed018bd&folder_id=b1giflegi2ke62rs80la: number of input tokens must be no more than 2048, got 2902"}"
	endpoint = "llm.api.cloud.yandex.net:443"
	stub_class = EmbeddingsServiceStub
	x-client-request-id = "13c4a031-8802-4fe2-abb6-976bbed018bd"
>

In [16]:
from sklearn.manifold import TSNE
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10
import numpy as np

# Convert embeddings to numpy array
embeddings_array = np.array(embeds)

# Perform T-SNE dimensionality reduction
tsne = TSNE(n_components=2, random_state=42)
embeddings_2d = tsne.fit_transform(embeddings_array)

# Prepare data for Bokeh
source = ColumnDataSource(data=dict(
    x=embeddings_2d[:, 0],
    y=embeddings_2d[:, 1],
    text=df['text'].head(1000).tolist()
))

# Create the plot
output_notebook()
p = figure(width=800, height=600, title="T-SNE Visualization of Text Embeddings")

# Add hover tool
hover = HoverTool(tooltips=[
    ("Text", "@text"),
])
p.add_tools(hover)

# Plot the points
p.circle('x', 'y', size=10, source=source, alpha=0.6)

# Show the plot
show(p)

Loading BokehJS ...

In [ ]:
# Add new cell for finding dense areas and visualizing top clusters
from sklearn.cluster import DBSCAN
from collections import Counter
import numpy as np
from bokeh.palettes import Spectral10
from bokeh.models import ColumnDataSource, HoverTool, Legend

# Perform DBSCAN clustering on the t-SNE results
clustering = DBSCAN(eps=3, min_samples=5).fit(embeddings_2d)
labels = clustering.labels_

# Count number of points in each cluster
cluster_counts = Counter(labels[labels != -1])  # Exclude noise points (-1)
top_clusters = sorted(cluster_counts.items(), key=lambda x: x[1], reverse=True)[:10]
top_cluster_labels = [label for label, _ in top_clusters]

# Create mask for points in top clusters
mask = np.isin(labels, top_cluster_labels)

# Prepare data for visualization
cluster_colors = {label: Spectral10[i] for i, (label, _) in enumerate(top_clusters)}
colors = ['#808080' if label == -1 else cluster_colors.get(label, '#808080') for label in labels]
sizes = [10 if label in top_cluster_labels else 5 for label in labels]
alphas = [0.8 if label in top_cluster_labels else 0.3 for label in labels]

# Create Bokeh data source
source = ColumnDataSource(data=dict(
    x=embeddings_2d[:, 0],
    y=embeddings_2d[:, 1],
    text=df['text'].head(len(embeddings_2d)).tolist(),
    color=colors,
    size=sizes,
    alpha=alphas,
    cluster=[f'Cluster {label}' if label in top_cluster_labels else 'Other' for label in labels]
))

# Create the plot
p = figure(width=1000, height=800, title="Top 10 Dense Question Clusters")

# Add hover tool
hover = HoverTool(tooltips=[
    ("Text", "@text"),
    ("Cluster", "@cluster")
])
p.add_tools(hover)

# Plot points with different colors and sizes
scatter = p.circle('x', 'y', size='size', color='color', alpha='alpha', source=source)

# Add legend
legend_items = [(f'Cluster {label} ({count} questions)', [scatter.select_by_value('cluster', f'Cluster {label}')]) 
                for label, count in top_clusters]
legend = Legend(items=legend_items, location='center_right')
p.add_layout(legend, 'right')

# Show the plot
show(p)

# Print statistics about top clusters
print("\nTop 10 densest clusters statistics:")
for label, count in top_clusters:
    cluster_texts = df['text'].head(len(embeddings_2d))[labels == label].tolist()
    print(f"\nCluster {label} ({count} questions):")
    print("Sample questions from this cluster:")
    for text in cluster_texts[:3]:  # Show first 3 questions from each cluster
        print(f"- {text[:200]}...")  # Show first 200 characters of each question
// ... existing code ...